# RQ1 GAD Reclassification Intermediate

This notebook applies the GitHub Advisory Database (GAD) correction after reviewing adjusted transparent lifecycle patterns. Cases whose fourth stage is `GAD` are reclassified as silent-equivalent lifecycle patterns because the security-fix link points to a vulnerability database entry rather than an issue/PR/report workflow.

Reclassification rules:

- `T4: Fix -> Release -> Disclosure -> GAD` becomes `S1: Fix -> Release -> Disclosure -> None`.
- `T6: Disclosure -> Fix -> Release -> GAD` becomes `S3: Disclosure -> Fix -> Release -> None`.
- `T7: Fix -> Disclosure -> Release -> GAD` becomes `S2: Fix -> Disclosure -> Release -> None`.

The original notebooks are left unchanged for traceability.


In [1]:
import contextlib
import io
import json
from pathlib import Path

import pandas as pd

In [2]:
FIX_RELEASE_PATH = Path('../../data/rq1/external_release_nvd/fix_releases_from_patch_data_local_server.csv')
NVD_PATH = Path('../../data/rq1/external_release_nvd/cve_NVD_disclosure_dates.csv')
LINKS_PATH = Path('../../data/rq1/corrected_resolved_links_v2.csv')
SEVERITY_PATH = Path('../../data/rq1/repo_with_severity.csv')
ADJUSTED_NOTEBOOK_PATH = Path('../../data/rq1/transparent_data/rq1_transparent_lifecycle_order_adjusted.ipynb')

fix_release = pd.read_csv(FIX_RELEASE_PATH)
nvd = pd.read_csv(NVD_PATH)
links = pd.read_csv(LINKS_PATH)
severity = pd.read_csv(SEVERITY_PATH)

print('fix_release:', fix_release.shape, 'unique CVEs:', fix_release['CVE_ID'].nunique())
print('nvd:', nvd.shape, 'unique CVEs:', nvd['CVE_ID'].nunique())
print('links:', links.shape, 'unique CVEs:', links['CVE_ID'].nunique())
print('severity:', severity.shape, 'unique CVEs:', severity['CVE_ID'].nunique())

fix_release: (832, 6) unique CVEs: 832
nvd: (832, 2) unique CVEs: 832
links: (832, 5) unique CVEs: 832
severity: (832, 10) unique CVEs: 832


## Build Original Silent Lifecycle Rows

This reproduces the silent lifecycle construction used in the existing RQ1 notebooks.


In [3]:
df = fix_release[fix_release['Oldest Tag Date'].astype(str).str.lower().ne('not found')].copy()

df['Fix Date'] = pd.to_datetime(df['Commit Date'], errors='coerce')
df['Release Date'] = pd.to_datetime(df['Oldest Tag Date'], errors='coerce')

df = df.merge(nvd[['CVE_ID', 'Published Date']], on='CVE_ID', how='left')
df['Disclosure Date'] = pd.to_datetime(df['Published Date'], errors='coerce')

df = df.merge(links[['CVE_ID', 'Link Presence']], on='CVE_ID', how='left')
df['Reporting characteristics'] = df['Link Presence'].map({
    'contains links': 'Transparent',
    'no links': 'Silent',
})

df = df.merge(severity[['CVE_ID', 'Severity']], on='CVE_ID', how='left')

event_df = df.dropna(subset=[
    'Fix Date',
    'Release Date',
    'Disclosure Date',
    'Reporting characteristics',
    'Severity',
]).copy()

release_before_fix_df = event_df[event_df['Release Date'] < event_df['Fix Date']].copy()
event_df = event_df[event_df['Release Date'] >= event_df['Fix Date']].copy()

print('Rows excluded because Release Date < Fix Date:', release_before_fix_df.shape[0])
print('Rows available for original lifecycle order table:', event_df.shape[0])
print('Unique CVEs:', event_df['CVE_ID'].nunique())

Rows excluded because Release Date < Fix Date: 15
Rows available for original lifecycle order table: 743
Unique CVEs: 743


In [4]:
def order_events(row):
    events = [
        ('Fix', row['Fix Date']),
        ('Release', row['Release Date']),
        ('Disclosure', row['Disclosure Date']),
    ]
    tie_breaker = {'Fix': 0, 'Release': 1, 'Disclosure': 2}
    ordered_names = [name for name, _ in sorted(events, key=lambda item: (item[1], tie_breaker[item[0]]))]
    if row['Reporting characteristics'] == 'Transparent':
        return ['Report'] + ordered_names
    return ordered_names + ['None']

ordered_columns = event_df.apply(order_events, axis=1, result_type='expand')
ordered_columns.columns = ['First', 'Second', 'Third', 'Fourth']

order_df = pd.concat([
    event_df[['CVE_ID', 'PATCH', 'Reporting characteristics', 'Severity', 'Fix Date', 'Release Date', 'Disclosure Date']].reset_index(drop=True),
    ordered_columns.reset_index(drop=True),
], axis=1)

pattern_ids = {
    ('Silent', 'Fix', 'Release', 'Disclosure', 'None'): 'S1',
    ('Silent', 'Fix', 'Disclosure', 'Release', 'None'): 'S2',
    ('Silent', 'Disclosure', 'Fix', 'Release', 'None'): 'S3',
    ('Transparent', 'Report', 'Fix', 'Release', 'Disclosure'): 'T1',
    ('Transparent', 'Report', 'Fix', 'Disclosure', 'Release'): 'T2',
    ('Transparent', 'Report', 'Disclosure', 'Fix', 'Release'): 'T3',
}

order_df['Pattern'] = order_df.apply(
    lambda row: pattern_ids.get((
        row['Reporting characteristics'], row['First'], row['Second'], row['Third'], row['Fourth']
    )),
    axis=1,
)

original_silent_order_df = order_df[order_df['Pattern'].isin(['S1', 'S2', 'S3'])].copy()
original_silent_order_df['Original reporting characteristics'] = 'Silent'
original_silent_order_df['Original pattern'] = original_silent_order_df['Pattern']
original_silent_order_df['Reclassification source'] = 'Original silent fix'
original_silent_order_df['Reclassification reason'] = ''

print('Original silent rows:', len(original_silent_order_df))
order_df['Pattern'].value_counts(dropna=False)

Original silent rows: 315


Pattern
T1    323
S1    258
T2     77
S2     38
T3     28
S3     19
Name: count, dtype: int64

## Load Adjusted Transparent Lifecycle Rows

The adjusted transparent lifecycle notebook contains full timestamp order and manual corrections for transparent cases with mined report dates.


In [5]:
adjusted_nb = json.loads(ADJUSTED_NOTEBOOK_PATH.read_text())
adjusted_ns = {}

captured_output = io.StringIO()
with contextlib.redirect_stdout(captured_output):
    for i, cell in enumerate(adjusted_nb['cells']):
        if cell.get('cell_type') == 'code':
            source = ''.join(cell.get('source', []))
            exec(compile(source, f'adjusted_notebook_cell_{i}', 'exec'), adjusted_ns)

transparent_lifecycle_order_adjusted_df = adjusted_ns['transparent_lifecycle_order_adjusted_df'].copy()
transparent_lifecycle_summary_adjusted = adjusted_ns['transparent_lifecycle_summary_adjusted'].copy()
manual_excluded_adjusted_df = adjusted_ns.get('manual_excluded_adjusted_df', pd.DataFrame()).copy()

print('Adjusted transparent lifecycle rows:', len(transparent_lifecycle_order_adjusted_df))
print('Manual excluded adjusted rows:', len(manual_excluded_adjusted_df))
transparent_lifecycle_summary_adjusted

Adjusted transparent lifecycle rows: 278
Manual excluded adjusted rows: 1


,First,Second,Third,Fourth,Count,%
0,Report,Fix,Release,Disclosure,216,77.70
1,Report,Fix,Disclosure,Release,26,9.35
2,Disclosure,Report,Fix,Release,13,4.68
3,Fix,Release,Disclosure,GAD,13,4.68
4,Report,Disclosure,Fix,Release,6,2.16
5,Disclosure,Fix,Release,GAD,2,0.72
6,Fix,Disclosure,Release,GAD,2,0.72


In [6]:
severity_order = ['Critical', 'High', 'Medium', 'Low', 'Unknown']
severity_levels = severity_order
severity_colors = {
    'Critical': '#d62728',
    'High': '#ff7f0e',
    'Medium': '#4c78a8',
    'Low': '#2ca25f',
    'Unknown': '#8c8c8c',
}

severity_lookup = severity[['CVE_ID', 'Severity']].drop_duplicates(subset=['CVE_ID']).copy()
transparent_order_df = transparent_lifecycle_order_adjusted_df.merge(
    severity_lookup,
    on='CVE_ID',
    how='left',
)

pattern_order_df = transparent_lifecycle_summary_adjusted.copy()
pattern_order_df['Pattern'] = [f'T{i}' for i in range(1, len(pattern_order_df) + 1)]
pattern_key = {
    (row['First'], row['Second'], row['Third'], row['Fourth']): row['Pattern']
    for _, row in pattern_order_df.iterrows()
}

transparent_order_df['Pattern'] = transparent_order_df.apply(
    lambda row: pattern_key[(row['First'], row['Second'], row['Third'], row['Fourth'])],
    axis=1,
)
transparent_order_df['Original reporting characteristics'] = 'Transparent'
transparent_order_df['Original pattern'] = transparent_order_df['Pattern']

print('Missing Severity:', transparent_order_df['Severity'].isna().sum())
pattern_order_df[['Pattern', 'First', 'Second', 'Third', 'Fourth', 'Count', '%']]

Missing Severity: 0


,Pattern,First,Second,Third,Fourth,Count,%
0,T1,Report,Fix,Release,Disclosure,216,77.70
1,T2,Report,Fix,Disclosure,Release,26,9.35
2,T3,Disclosure,Report,Fix,Release,13,4.68
3,T4,Fix,Release,Disclosure,GAD,13,4.68
4,T5,Report,Disclosure,Fix,Release,6,2.16
5,T6,Disclosure,Fix,Release,GAD,2,0.72
6,T7,Fix,Disclosure,Release,GAD,2,0.72


## Reclassify GAD-Ending Transparent Patterns


In [7]:
gad_pattern_to_silent_pattern = {
    ('Fix', 'Release', 'Disclosure', 'GAD'): 'S1',
    ('Fix', 'Disclosure', 'Release', 'GAD'): 'S2',
    ('Disclosure', 'Fix', 'Release', 'GAD'): 'S3',
}

transparent_order_df['Adjusted order tuple'] = list(zip(
    transparent_order_df['First'],
    transparent_order_df['Second'],
    transparent_order_df['Third'],
    transparent_order_df['Fourth'],
))

reclassified_gad_df = transparent_order_df[
    transparent_order_df['Adjusted order tuple'].isin(gad_pattern_to_silent_pattern)
].copy()
reclassified_gad_df['Pattern'] = reclassified_gad_df['Adjusted order tuple'].map(gad_pattern_to_silent_pattern)
reclassified_gad_df['Reporting characteristics'] = 'Silent'
reclassified_gad_df['Fourth'] = 'None'
reclassified_gad_df['Reclassification source'] = 'GAD-ending adjusted transparent pattern'
reclassified_gad_df['Reclassification reason'] = 'GAD is a vulnerability database link similar to NVD, not an issue/PR/report workflow.'

corrected_transparent_order_df = transparent_order_df[
    ~transparent_order_df['Adjusted order tuple'].isin(gad_pattern_to_silent_pattern)
].copy()
corrected_transparent_order_df['Reporting characteristics'] = 'Transparent'
corrected_transparent_order_df['Reclassification source'] = 'Adjusted transparent fix retained'
corrected_transparent_order_df['Reclassification reason'] = ''

silent_columns = [
    'CVE_ID', 'PATCH', 'Reporting characteristics', 'Severity',
    'Fix Date', 'Release Date', 'Disclosure Date',
    'First', 'Second', 'Third', 'Fourth', 'Pattern',
    'Original reporting characteristics', 'Original pattern',
    'Reclassification source', 'Reclassification reason',
]

original_silent_for_concat = original_silent_order_df[silent_columns].copy()
reclassified_gad_for_concat = reclassified_gad_df[silent_columns].copy()
corrected_silent_order_df = pd.concat([
    original_silent_for_concat,
    reclassified_gad_for_concat,
], ignore_index=True)

corrected_transparent_pattern_order = [
    pattern for pattern in pattern_order_df['Pattern'].tolist()
    if pattern in set(corrected_transparent_order_df['Pattern'])
]

corrected_transparent_pattern_titles = {
    row['Pattern']: f"{row['Pattern']}: {row['First']} -> {row['Second']} -> {row['Third']} -> {row['Fourth']} (n={int(corrected_transparent_order_df['Pattern'].eq(row['Pattern']).sum())})"
    for _, row in pattern_order_df.iterrows()
    if row['Pattern'] in corrected_transparent_pattern_order
}

print('Original silent rows:', len(original_silent_order_df))
print('GAD rows reclassified to silent:', len(reclassified_gad_df))
print('Corrected silent rows:', len(corrected_silent_order_df))
print('Corrected transparent rows:', len(corrected_transparent_order_df))
print('Corrected transparent pattern order:', corrected_transparent_pattern_order)

Original silent rows: 315
GAD rows reclassified to silent: 17
Corrected silent rows: 332
Corrected transparent rows: 261
Corrected transparent pattern order: ['T1', 'T2', 'T3', 'T5']


In [8]:
def format_severity_distribution(values):
    counts = values.value_counts(dropna=False).to_dict()
    parts = []
    for severity_label in severity_order:
        count = int(counts.get(severity_label, 0))
        if severity_label == 'Unknown':
            if count > 0:
                parts.append(f'{severity_label} ({count})')
        else:
            parts.append(f'{severity_label} ({count})')
    for severity_label, count in sorted(counts.items(), key=lambda item: str(item[0])):
        if severity_label not in severity_order:
            parts.append(f'{severity_label} ({int(count)})')
    return ', '.join(parts)


def median_iqr_text(values):
    values = values.dropna()
    if values.empty:
        return 'NA'
    return f'{values.median():.2f} [{values.quantile(0.25):.2f}, {values.quantile(0.75):.2f}]'


def date_for_event(row, event_name):
    return row[f'{event_name} Date']


def add_three_stage_durations(df):
    out = df.copy()
    out['First Date'] = out.apply(lambda row: date_for_event(row, row['First']), axis=1)
    out['Second Date'] = out.apply(lambda row: date_for_event(row, row['Second']), axis=1)
    out['Third Date'] = out.apply(lambda row: date_for_event(row, row['Third']), axis=1)
    out['Days: First -> Second'] = (out['Second Date'] - out['First Date']).dt.total_seconds() / 86400
    out['Days: Second -> Third'] = (out['Third Date'] - out['Second Date']).dt.total_seconds() / 86400
    out['Days: First -> Third'] = (out['Third Date'] - out['First Date']).dt.total_seconds() / 86400
    return out


def add_four_stage_durations(df):
    out = df.copy()
    out['First Date'] = out.apply(lambda row: date_for_event(row, row['First']), axis=1)
    out['Second Date'] = out.apply(lambda row: date_for_event(row, row['Second']), axis=1)
    out['Third Date'] = out.apply(lambda row: date_for_event(row, row['Third']), axis=1)
    out['Fourth Date'] = out.apply(lambda row: date_for_event(row, row['Fourth']), axis=1)
    out['Days: First -> Second'] = (out['Second Date'] - out['First Date']).dt.total_seconds() / 86400
    out['Days: Second -> Third'] = (out['Third Date'] - out['Second Date']).dt.total_seconds() / 86400
    out['Days: Third -> Fourth'] = (out['Fourth Date'] - out['Third Date']).dt.total_seconds() / 86400
    out['Days: First -> Fourth'] = (out['Fourth Date'] - out['First Date']).dt.total_seconds() / 86400
    return out

corrected_silent_order_df = add_three_stage_durations(corrected_silent_order_df)
corrected_transparent_order_df = add_four_stage_durations(corrected_transparent_order_df)

print('Corrected silent duration rows:', len(corrected_silent_order_df))
print('Corrected transparent duration rows:', len(corrected_transparent_order_df))

Corrected silent duration rows: 332
Corrected transparent duration rows: 261


## Reclassification Audit


In [9]:
gad_reclassification_audit = (
    reclassified_gad_df
    .groupby(['Original pattern', 'First', 'Second', 'Third', 'Fourth', 'Pattern'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{'Severity distribution': ('Severity', format_severity_distribution)}
    )
    .reset_index()
    .rename(columns={
        'Original pattern': 'Original transparent pattern',
        'Pattern': 'Corrected silent pattern',
    })
)

gad_reclassification_audit

,Original transparent pattern,First,Second,Third,Fourth,Corrected silent pattern,Count,Severity distribution
0,T4,Fix,Release,Disclosure,None,S1,13,"Critical (1), High (2), Medium (9), Low (1)"
1,T6,Disclosure,Fix,Release,None,S3,2,"Critical (0), High (0), Medium (2), Low (0)"
2,T7,Fix,Disclosure,Release,None,S2,2,"Critical (0), High (0), Medium (2), Low (0)"


## Corrected Silent Tables


In [10]:
corrected_silent_timing_table = (
    corrected_silent_order_df
    .groupby(['Reporting characteristics', 'Pattern', 'First', 'Second', 'Third', 'Fourth'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{
            'Median days: First -> Second': ('Days: First -> Second', 'median'),
            'Median days: Second -> Third': ('Days: Second -> Third', 'median'),
            'Median days: First -> Third': ('Days: First -> Third', 'median'),
            'Severity distribution': ('Severity', format_severity_distribution),
        }
    )
    .reset_index()
)
corrected_silent_timing_table['%'] = (
    corrected_silent_timing_table['Count'] / corrected_silent_timing_table['Count'].sum() * 100
).round(2)
for column in ['Median days: First -> Second', 'Median days: Second -> Third', 'Median days: First -> Third']:
    corrected_silent_timing_table[column] = corrected_silent_timing_table[column].round(2)
corrected_silent_timing_table = corrected_silent_timing_table[
    [
        'Reporting characteristics', 'Pattern', 'First', 'Second', 'Third', 'Fourth',
        'Count', '%', 'Median days: First -> Second', 'Median days: Second -> Third',
        'Median days: First -> Third', 'Severity distribution'
    ]
].sort_values('Pattern').reset_index(drop=True)

corrected_silent_timing_table

,Reporting characteristics,Pattern,First,Second,Third,Fourth,Count,%,Median days: First -> Second,Median days: Second -> Third,Median days: First -> Third,Severity distribution
0,Silent,S1,Fix,Release,Disclosure,None,271,81.63,5.96,26.99,45.42,"Critical (7), High (40), Medium (211), Low (12..."
1,Silent,S2,Fix,Disclosure,Release,None,40,12.05,16.90,35.97,86.54,"Critical (1), High (7), Medium (30), Low (2)"
2,Silent,S3,Disclosure,Fix,Release,None,21,6.33,20.90,14.84,31.60,"Critical (1), High (0), Medium (19), Low (1)"


In [11]:
corrected_silent_severity_timing_table = (
    corrected_silent_order_df
    .groupby(['Reporting characteristics', 'Pattern', 'Severity', 'First', 'Second', 'Third', 'Fourth'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{
            'First -> Second days, median [IQR]': ('Days: First -> Second', median_iqr_text),
            'Second -> Third days, median [IQR]': ('Days: Second -> Third', median_iqr_text),
            'First -> Third days, median [IQR]': ('Days: First -> Third', median_iqr_text),
        }
    )
    .reset_index()
)
corrected_silent_severity_timing_table['Pattern total'] = corrected_silent_severity_timing_table.groupby('Pattern')['Count'].transform('sum')
corrected_silent_severity_timing_table['Within-pattern %'] = (
    corrected_silent_severity_timing_table['Count'] / corrected_silent_severity_timing_table['Pattern total'] * 100
).round(2)
severity_rank = {severity: index for index, severity in enumerate(severity_order)}
corrected_silent_severity_timing_table['Severity rank'] = corrected_silent_severity_timing_table['Severity'].map(severity_rank).fillna(99)
corrected_silent_severity_timing_table = (
    corrected_silent_severity_timing_table
    .sort_values(['Pattern', 'Severity rank'])
    .drop(columns=['Pattern total', 'Severity rank'])
    .reset_index(drop=True)
)
corrected_silent_severity_timing_table

,Reporting characteristics,Pattern,Severity,First,Second,Third,Fourth,Count,"First -> Second days, median [IQR]","Second -> Third days, median [IQR]","First -> Third days, median [IQR]",Within-pattern %
0,Silent,S1,Critical,Fix,Release,Disclosure,None,7,"5.96 [0.15, 10.50]","8.13 [6.37, 24.49]","12.06 [7.50, 62.16]",2.58
1,Silent,S1,High,Fix,Release,Disclosure,None,40,"2.56 [0.19, 21.33]","35.32 [10.73, 251.31]","47.68 [17.96, 310.56]",14.76
2,Silent,S1,Medium,Fix,Release,Disclosure,None,211,"6.88 [1.01, 28.01]","28.61 [6.60, 153.84]","49.60 [17.32, 214.80]",77.86
3,Silent,S1,Low,Fix,Release,Disclosure,None,12,"6.06 [0.84, 10.11]","15.36 [8.60, 44.95]","20.15 [13.67, 50.09]",4.43
4,Silent,S1,Unknown,Fix,Release,Disclosure,None,1,"0.03 [0.03, 0.03]","795.43 [795.43, 795.43]","795.46 [795.46, 795.46]",0.37
5,Silent,S2,Critical,Fix,Disclosure,Release,None,1,"3.39 [3.39, 3.39]","104.55 [104.55, 104.55]","107.93 [107.93, 107.93]",2.50
6,Silent,S2,High,Fix,Disclosure,Release,None,7,"6.52 [3.04, 13.46]","56.93 [19.53, 193.90]","77.49 [32.74, 196.93]",17.50
7,Silent,S2,Medium,Fix,Disclosure,Release,None,30,"33.60 [10.07, 76.93]","31.92 [11.96, 188.77]","90.15 [51.96, 285.46]",75.00
8,Silent,S2,Low,Fix,Disclosure,Release,None,2,"10.19 [8.08, 12.29]","11.58 [8.39, 14.76]","21.76 [16.47, 27.06]",5.00
9,Silent,S3,Critical,Disclosure,Fix,Release,None,1,"8.39 [8.39, 8.39]","1.25 [1.25, 1.25]","9.64 [9.64, 9.64]",4.76


## Corrected Transparent Tables


In [12]:
corrected_transparent_lifecycle_table = (
    corrected_transparent_order_df
    .groupby(['Reporting characteristics', 'Pattern', 'First', 'Second', 'Third', 'Fourth'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{'Severity distribution': ('Severity', format_severity_distribution)}
    )
    .reset_index()
)
corrected_transparent_lifecycle_table['%'] = (
    corrected_transparent_lifecycle_table['Count'] / corrected_transparent_lifecycle_table['Count'].sum() * 100
).round(2)
corrected_transparent_lifecycle_table['Pattern'] = pd.Categorical(
    corrected_transparent_lifecycle_table['Pattern'],
    categories=corrected_transparent_pattern_order,
    ordered=True,
)
corrected_transparent_lifecycle_table = corrected_transparent_lifecycle_table.sort_values('Pattern').reset_index(drop=True)
corrected_transparent_lifecycle_table

,Reporting characteristics,Pattern,First,Second,Third,Fourth,Count,Severity distribution,%
0,Transparent,T1,Report,Fix,Release,Disclosure,216,"Critical (10), High (42), Medium (147), Low (17)",82.76
1,Transparent,T2,Report,Fix,Disclosure,Release,26,"Critical (1), High (7), Medium (16), Low (2)",9.96
2,Transparent,T3,Disclosure,Report,Fix,Release,13,"Critical (2), High (5), Medium (6), Low (0)",4.98
3,Transparent,T5,Report,Disclosure,Fix,Release,6,"Critical (0), High (1), Medium (5), Low (0)",2.30


In [13]:
corrected_transparent_stage_timing_table = (
    corrected_transparent_order_df
    .groupby(['Pattern', 'Severity', 'First', 'Second', 'Third', 'Fourth'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{
            'First -> Second days, median [IQR]': ('Days: First -> Second', median_iqr_text),
            'Second -> Third days, median [IQR]': ('Days: Second -> Third', median_iqr_text),
            'Third -> Fourth days, median [IQR]': ('Days: Third -> Fourth', median_iqr_text),
            'First -> Fourth days, median [IQR]': ('Days: First -> Fourth', median_iqr_text),
        }
    )
    .reset_index()
)
corrected_transparent_stage_timing_table['Pattern'] = pd.Categorical(
    corrected_transparent_stage_timing_table['Pattern'], categories=corrected_transparent_pattern_order, ordered=True
)
corrected_transparent_stage_timing_table['Severity'] = pd.Categorical(
    corrected_transparent_stage_timing_table['Severity'], categories=severity_order, ordered=True
)
corrected_transparent_stage_timing_table = corrected_transparent_stage_timing_table.sort_values(['Pattern', 'Severity']).reset_index(drop=True)
corrected_transparent_stage_timing_table

,Pattern,Severity,First,Second,Third,Fourth,Count,"First -> Second days, median [IQR]","Second -> Third days, median [IQR]","Third -> Fourth days, median [IQR]","First -> Fourth days, median [IQR]"
0,T1,Critical,Report,Fix,Release,Disclosure,10,"2.53 [0.13, 11.57]","3.82 [0.50, 10.05]","10.86 [5.00, 27.04]","35.25 [12.90, 154.65]"
1,T1,High,Report,Fix,Release,Disclosure,42,"0.34 [0.08, 8.20]","11.32 [1.95, 38.55]","59.22 [20.52, 183.90]","82.52 [41.91, 240.61]"
2,T1,Medium,Report,Fix,Release,Disclosure,147,"2.00 [0.13, 18.12]","5.89 [0.42, 26.42]","29.64 [9.06, 84.01]","65.24 [21.43, 182.18]"
3,T1,Low,Report,Fix,Release,Disclosure,17,"4.72 [1.37, 12.46]","10.05 [5.46, 43.34]","65.29 [7.45, 138.39]","101.41 [52.44, 201.78]"
4,T2,Critical,Report,Fix,Disclosure,Release,1,"708.26 [708.26, 708.26]","52.92 [52.92, 52.92]","0.00 [0.00, 0.00]","761.18 [761.18, 761.18]"
5,T2,High,Report,Fix,Disclosure,Release,7,"3.54 [1.48, 22.57]","13.48 [6.85, 49.88]","53.86 [45.68, 125.44]","154.04 [103.31, 164.78]"
6,T2,Medium,Report,Fix,Disclosure,Release,16,"1.15 [0.21, 2.23]","29.89 [7.37, 47.06]","63.63 [15.53, 134.78]","161.42 [41.06, 249.58]"
7,T2,Low,Report,Fix,Disclosure,Release,2,"0.32 [0.22, 0.41]","7.60 [7.12, 8.07]","11.69 [6.18, 17.20]","19.60 [13.52, 25.68]"
8,T3,Critical,Disclosure,Report,Fix,Release,2,"1.79 [1.28, 2.30]","3.36 [2.19, 4.53]","134.72 [90.40, 179.03]","139.87 [94.90, 184.85]"
9,T3,High,Disclosure,Report,Fix,Release,5,"14.81 [3.40, 66.02]","0.86 [0.62, 1.02]","3.64 [2.68, 46.09]","61.92 [6.36, 70.52]"


## Optional Exports

No CSVs are saved by default.


In [14]:
# Uncomment if needed.
# corrected_silent_order_df.to_csv('corrected_silent_order_rows_after_gad_reclassification.csv', index=False)
# corrected_transparent_order_df.to_csv('corrected_transparent_order_rows_after_gad_reclassification.csv', index=False)
# corrected_silent_timing_table.to_csv('corrected_silent_timing_table_after_gad_reclassification.csv', index=False)
# corrected_transparent_lifecycle_table.to_csv('corrected_transparent_lifecycle_table_after_gad_reclassification.csv', index=False)